In [22]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from datetime import time as dt_time
from typing import List, Tuple, Dict, Optional
import os

from ortools.linear_solver import pywraplp


In [23]:
Rooms = pd.read_excel("../data/Raw Data/Data.xlsx" , sheet_name="Rooms")
Courses = pd.read_excel("../data/Raw Data/Data.xlsx" , sheet_name="Courses" )
Doctors = pd.read_excel("../data/Raw Data/Data.xlsx" , sheet_name="Doctors" )
Divisions = pd.read_excel("../data/Raw Data/Data.xlsx" , sheet_name="Division" )

In [24]:
Rooms.head(10)

,Room_ID,Room,Capacity,Type
0,R1,Room 1,30,Lab
1,R2,Room 2,30,Lab
2,R3,Room 3,30,Lab
3,R4,Room 4,30,Lab
4,R5,Room 5,30,Lab
5,T8,Terrace 8,250,Lecture
6,T4,terrace4,100,Lecture
7,H16,Hall 16,100,Lecture


In [25]:
Courses.head(10)

,Course_ID,Course_Name,Department,Major,Days,Hours_per_day,Instructor_ID,Year,Type,Duration
0,C01,Mathematics (0),IT,IT,2,1,I12,1,Lecture,1 Hour
1,C02,Mathematics (1),IT,IT,2,1,I12,1,Lecture,1 Hour
2,C03,Electronics,IT,IT,2,1,I13,1,Lecture,1 Hour
3,C04,Discrete Mathematics,IT,IT,2,1,I16,1,Lecture,1 Hour
4,C05,Introduction to Computers,IT,IT,2,1,I18,1,Lecture,1 Hour
5,C06,Mathematics-3,IT,IT,2,1,I16,2,Lecture,1 Hour
6,C07,Computer Networks Technology,IT,IT,2,1,I14,2,Lecture,1 Hour
7,C08,Introduction to Software Engineering,IT,IT,2,1,I15,2,Lecture,1 Hour
8,C09,Object Oriented Programming,IT,IT,2,1,I03,2,Lecture,1 Hour
9,C10,Probability and Statistics-2,IT,IT,2,1,I05,2,Lecture,1 Hour


In [26]:
Doctors.head(10)
def prepare_pairs(self):

        self.pairs = []

        for _, course in self.Courses.iterrows():

            matching = self.Divisions[
                (self.Divisions["Year"] == course["Year"]) &
                (self.Divisions["Major"] == course["Major"]) &
                (self.Divisions["Department"] == course["Department"])
            ]

            for _, div in matching.iterrows():
                self.pairs.append((course, div))

def run(self):

        solver = pywraplp.Solver.CreateSolver("CBC")
        x = {}

        # VARIABLES
        for course, div in self.pairs:

            duration = course["Hours_per_day"] * 60
            instructor = course["Instructor_ID"]

            for day in days:
                for start in time_slots:

                    end = start + duration
                    if end > 17*60:
                        continue

                    for _, room in self.Rooms.iterrows():

                        if room["Type"] != course["Type"]:
                            continue

                        if room["Capacity"] < div["StudentNum"]:
                            continue

                        key = (
                            course["Course_ID"],
                            div["Num_ID"],
                            instructor,
                            day,
                            start,
                            end,
                            room["Room_ID"]
                        )

                        x[key] = solver.IntVar(0, 1, str(key))

        print("Variables:", len(x))

        # REQUIRED DAYS (≤ مش == عشان يطلع جدول)
        for course, div in self.pairs:

            required = course["Days"]

            relevant = [
                var for key, var in x.items()
                if key[0] == course["Course_ID"]
                and key[1] == div["Num_ID"]
            ]

            if relevant:
                solver.Add(solver.Sum(relevant) <= required)

        # CONFLICTS
        keys = list(x.keys())

        for i in range(len(keys)):
            for j in range(i+1, len(keys)):

                k1 = keys[i]
                k2 = keys[j]

                if k1[3] != k2[3]:
                    continue

                if not overlap(k1[4], k1[5], k2[4], k2[5]):
                    continue

                # Room conflict
                if k1[6] == k2[6]:
                    solver.Add(x[k1] + x[k2] <= 1)

                # Instructor conflict
                if k1[2] == k2[2]:
                    solver.Add(x[k1] + x[k2] <= 1)

                # Division conflict
                if k1[1] == k2[1]:
                    solver.Add(x[k1] + x[k2] <= 1)

        solver.Maximize(solver.Sum(x.values()))
        solver.SetTimeLimit(60000)

        status = solver.Solve()

        if status not in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
            print("No Solution Found")
            return []

        schedule = []

        for key, var in x.items():
            if var.solution_value() == 1:
                schedule.append(key)

        return schedule


In [27]:
Divisions.head(20)

,Num_ID,Department,Major,Year,StudentNum
0,D01,IT,IT,1,219
1,D02,IT,IT,2,235
2,D03,IT,IT,3,264
3,D04,IT,IT,4,299
4,D05,BA,BA,1,125
5,D06,BA,BA,2,127
6,D07,BA,BA,3,65
7,D08,BA,AC,3,117
8,D09,BA,BA,4,54
9,D10,BA,AC,4,77


In [28]:
def parse_time(t):
    h, m = map(int, str(t).split(":"))
    return h * 60 + m


def time_ranges_overlap(start1, end1, start2, end2):
    return not (end1 <= start2 or end2 <= start1)


def parse_time_extended(time_str):
    try:
        if isinstance(time_str, str):
            time_str = time_str.strip().upper()

            is_pm = "PM" in time_str
            is_am = "AM" in time_str

            time_str = time_str.replace("AM", "").replace("PM", "").strip()

            parts = time_str.split(":")
            hours = int(parts[0])
            minutes = int(parts[1]) if len(parts) > 1 else 0

            if hours == 12:
                if is_am:
                    hours = 0
            elif is_pm and hours != 12:
                hours += 12

            return hours * 60 + minutes
        return 0
    except:
        return 0


def minutes_to_time_str(minutes):
    if minutes == 0:
        return "12:00 AM"
    hours = minutes // 60
    mins = minutes % 60
    hours = hours % 24

    if hours == 0:
        hour_12 = 12
        period = "AM"
    elif hours < 12:
        hour_12 = hours
        period = "AM"
    elif hours == 12:
        hour_12 = 12
        period = "PM"
    else:
        hour_12 = hours - 12
        period = "PM"

    return f"{hour_12}:{mins:02d} {period}"


def is_time_in_range(time_minutes, start_minutes, end_minutes):
    if end_minutes < start_minutes:
        return time_minutes >= start_minutes or time_minutes <= end_minutes
    return start_minutes <= time_minutes <= end_minutes


def time_to_minutes(time_value):
    if isinstance(time_value, (int, float)):
        return int(time_value)

    if isinstance(time_value, dt_time):
        return time_value.hour * 60 + time_value.minute

    if isinstance(time_value, str):
        result = parse_time_extended(time_value)
        if result > 0:
            return result

        try:
            return parse_time(time_value)
        except:
            return 0

    return 0


print("Helper functions defined")

Helper functions defined


In [29]:
# Prepare data structures
room_dict = {row['Room_ID']: {'capacity': row['Capacity'], 'type': row['Type']} 
             for _, row in Rooms.iterrows()}

course_dict = {row['Course_ID']: {
    'instructor': row['Instructor_ID'],
    'days': row['Days'],
    'hours_per_day': row['Hours_per_day'],
    'type': row['Type'],
    'year': row['Year'],
    'major': row['Major'],
    'department': row['Department']
} for _, row in Courses.iterrows()}

doctor_availability = {}
for _, row in Doctors.iterrows():
    inst_id = row['Instructor_ID']
    day = row['Day']
    start = row['Start_Time']
    end = row['End_Time']
    start_minutes = time_to_minutes(start)
    end_minutes = time_to_minutes(end)
    
    if inst_id not in doctor_availability:
        doctor_availability[inst_id] = {}
    if day not in doctor_availability[inst_id]:
        doctor_availability[inst_id][day] = []
    doctor_availability[inst_id][day].append((start_minutes, end_minutes))

division_dict = {row['Num_ID']: {
    'students': row['StudentNum'],
    'year': row['Year'],
    'major': row['Major'],
    'department': row['Department']
} for _, row in Divisions.iterrows()}

lecture_rooms = [r for r in room_dict.keys() if room_dict[r]['type'] == 'Lecture']
lab_rooms = [r for r in room_dict.keys() if room_dict[r]['type'] == 'Lab']

print(f"Data prepared: {len(course_dict)} courses, {len(room_dict)} rooms, {len(division_dict)} divisions")
print(f"Lecture rooms: {len(lecture_rooms)}, Lab rooms: {len(lab_rooms)}")

Data prepared: 53 courses, 8 rooms, 10 divisions
Lecture rooms: 3, Lab rooms: 5


In [30]:
days = Doctors['Day'].unique().tolist()
time_slots = list(range(8 * 60, 17 * 60 + 1, 60))


def overlap(s1, e1, s2, e2):
    return max(s1, s2) < min(e1, e2)


class SchedulingILP:
    """Integer Linear Programming scheduler using OR-Tools CBC solver."""

    def __init__(self, courses_df, rooms_df, divisions_df, time_limit_millis: int = 60000):
        self.courses_df = courses_df
        self.rooms_df = rooms_df
        self.divisions_df = divisions_df
        self.time_limit_millis = time_limit_millis
        self.prepare_pairs()

    def prepare_pairs(self):
        """Prepare all (course, division) pairs, matching by year/major/department like CP notebook."""
        self.pairs = []

        for _, course in self.courses_df.iterrows():
            year = course["Year"]
            major = course["Major"]
            dept = course["Department"]

            matching = self.divisions_df[
                (self.divisions_df["Year"] == year)
                & (self.divisions_df["Major"] == major)
                & (self.divisions_df["Department"] == dept)
            ]

            if not matching.empty:
                for _, div in matching.iterrows():
                    self.pairs.append((course, div))

    def run(self):
        solver = pywraplp.Solver.CreateSolver("CBC")
        x: Dict[tuple, pywraplp.Variable] = {}

        # Decision variables
        for course, div in self.pairs:
            duration = int(course["Hours_per_day"]) * 60
            instructor = course["Instructor_ID"]

            for day in days:
                # Instructor must be available that day (if we have data)
                if instructor in doctor_availability and day not in doctor_availability[instructor]:
                    continue

                for start in time_slots:
                    end = start + duration
                    if end > 17 * 60:
                        continue

                    for _, room in self.rooms_df.iterrows():
                        if room["Type"] != course["Type"]:
                            continue

                        # Use half the students for room capacity check to match CP behavior
                        total_students = int(div["StudentNum"])
                        students_requiring_room = total_students // 2
                        if room["Capacity"] < students_requiring_room:
                            continue

                        key = (
                            course["Course_ID"],
                            div["Num_ID"],
                            instructor,
                            day,
                            start,
                            end,
                            room["Room_ID"],
                        )

                        x[key] = solver.IntVar(0, 1, str(key))

        print("Variables:", len(x))

        # Each course-division pair must meet its required number of days (at most)
        for course, div in self.pairs:
            required = int(course["Days"])

            relevant = [
                var
                for key, var in x.items()
                if key[0] == course["Course_ID"] and key[1] == div["Num_ID"]
            ]

            if relevant:
                solver.Add(solver.Sum(relevant) <= required)

        # Conflict constraints (room, instructor, division)
        keys = list(x.keys())

        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                k1 = keys[i]
                k2 = keys[j]

                # Different day -> no conflict
                if k1[3] != k2[3]:
                    continue

                # No time overlap -> no conflict
                if not overlap(k1[4], k1[5], k2[4], k2[5]):
                    continue

                # Room conflict
                if k1[6] == k2[6]:
                    solver.Add(x[k1] + x[k2] <= 1)

                # Instructor conflict
                if k1[2] == k2[2]:
                    solver.Add(x[k1] + x[k2] <= 1)

                # Division conflict
                if k1[1] == k2[1]:
                    solver.Add(x[k1] + x[k2] <= 1)

        # Objective: maximize number of assigned course slots
        solver.Maximize(solver.Sum(x.values()))
        solver.SetTimeLimit(self.time_limit_millis)

        status = solver.Solve()

        if status not in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
            print("No Solution Found")
            return []

        schedule = []
        for key, var in x.items():
            if var.solution_value() == 1:
                schedule.append(key)

        return schedule


print("ILP scheduler class defined")

ILP scheduler class defined


In [31]:
# Create and run ILP scheduler
ilp_scheduler = SchedulingILP(
    courses_df=Courses,
    rooms_df=Rooms,
    divisions_df=Divisions,
    time_limit_millis=60000,
)

result = ilp_scheduler.run()
print("Assignments:", len(result))

Variables: 2871
Assignments: 104


In [32]:
# Build human-readable schedule DataFrame
schedule_data = []

for r in result:
    course_id, div_id, instructor, day, start, end, room_id = r

    course_info = Courses[Courses["Course_ID"] == course_id].iloc[0]
    instructor_info = Doctors[Doctors["Instructor_ID"] == instructor].iloc[0]
    division_info = Divisions[Divisions["Num_ID"] == div_id].iloc[0]
    room_info = Rooms[Rooms["Room_ID"] == room_id].iloc[0]

    start_time_str = minutes_to_time_str(start)
    end_time_str = minutes_to_time_str(end)

    schedule_data.append({
        "Day": day,
        "Course_Name": course_info["Course_Name"],
        "Instructor_Name": instructor_info["Instructor_Name"],
        "Students": int(division_info["StudentNum"]) // 2,
        "Room": room_info["Room"],
        "Start_Time": start_time_str,
        "End_Time": end_time_str,
        "Department": course_info["Department"],
        "Major": course_info["Major"],
    })

Schedule = pd.DataFrame(schedule_data)

print(f"\nGenerated schedule with {len(Schedule)} assignments:")
print(Schedule.head(20))
print(f"\nTotal assignments: {len(Schedule)}")


Generated schedule with 104 assignments:
          Day                           Course_Name     Instructor_Name  \
0     Tuesday                       Mathematics (0)           Dr. Amany   
1     Tuesday                       Mathematics (0)           Dr. Amany   
2    Saturday                       Mathematics (1)           Dr. Amany   
3     Tuesday                       Mathematics (1)           Dr. Amany   
4     Tuesday                          Electronics          Dr. Ibrahim   
5     Tuesday                          Electronics          Dr. Ibrahim   
6      Sunday                 Discrete Mathematics           Dr. Moataz   
7   Wednesday                 Discrete Mathematics           Dr. Moataz   
8      Sunday             Introduction to Computers     Dr. Dina Tarek    
9   Wednesday             Introduction to Computers     Dr. Dina Tarek    
10     Sunday                        Mathematics-3           Dr. Moataz   
11     Sunday                        Mathematics-3        

In [33]:
# Verify room capacity for each assignment (like CP notebook)
print("\n--- Room capacity verification ---")
over_count = 0
for r in result:
    course_id, div_id, instructor, day, start, end, room_id = r
    cap = room_dict.get(room_id, {}).get('capacity', 0)
    div_info = Divisions[Divisions['Num_ID'] == div_id].iloc[0]
    students = int(div_info['StudentNum']) // 2  # half for capacity check
    if cap < students:
        over_count += 1
        print(f"  OVER: {course_id} -> {room_id} (capacity {cap}, students {students})")
if over_count == 0:
    print("  All assignments respect room capacity.")
else:
    print(f"  Warning: {over_count} assignment(s) exceed room capacity.")

# Day checker: 1-day courses must have 1 day, 2-day courses must have 2 distinct days
print("\n--- Day checker ---")
from collections import defaultdict
course_to_days = defaultdict(set)
course_to_assignments = defaultdict(int)
for r in result:
    course_id, div_id, instructor, day, start, end, room_id = r
    course_to_days[course_id].add(day)
    course_to_assignments[course_id] += 1

day_ok = True
for course_id in sorted(course_to_days.keys()):
    required = int(Courses[Courses['Course_ID'] == course_id]['Days'].iloc[0])
    distinct_days = len(course_to_days[course_id])
    num_slots = course_to_assignments[course_id]
    if distinct_days != required or num_slots != required:
        day_ok = False
        print(f"  BAD: {course_id} required {required} day(s), got {num_slots} slot(s) on {distinct_days} day(s): {sorted(course_to_days[course_id])}")
if day_ok:
    print("  All courses have correct number of days (1-day = 1 day, 2-day = 2 days).")


--- Room capacity verification ---
  All assignments respect room capacity.

--- Day checker ---
  BAD: C01 required 2 day(s), got 2 slot(s) on 1 day(s): ['Tuesday']
  BAD: C03 required 2 day(s), got 2 slot(s) on 1 day(s): ['Tuesday']
  BAD: C06 required 2 day(s), got 2 slot(s) on 1 day(s): ['Sunday']
  BAD: C11 required 2 day(s), got 2 slot(s) on 1 day(s): ['Monday']
  BAD: C12 required 2 day(s), got 2 slot(s) on 1 day(s): ['Saturday']
  BAD: C14 required 2 day(s), got 2 slot(s) on 1 day(s): ['Saturday']
  BAD: C15 required 2 day(s), got 1 slot(s) on 1 day(s): ['Saturday']
  BAD: C17 required 2 day(s), got 2 slot(s) on 1 day(s): ['Wednesday']
  BAD: C24 required 2 day(s), got 2 slot(s) on 1 day(s): ['Thursday']
  BAD: C25 required 2 day(s), got 2 slot(s) on 1 day(s): ['Monday']
  BAD: C26 required 2 day(s), got 2 slot(s) on 1 day(s): ['Monday']
  BAD: C33 required 2 day(s), got 2 slot(s) on 1 day(s): ['Sunday']
  BAD: C34 required 2 day(s), got 2 slot(s) on 1 day(s): ['Thursday']
  B

In [34]:
# Save schedule to this folder (ILP version)
OUTPUT_DIR = r"C:\Work\ML\Graduation\data\Processed Data"

if result:
    try:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        output_path = os.path.join(OUTPUT_DIR, 'Schedule_Output_ILP.xlsx')
        Schedule.to_excel(output_path, sheet_name='Schedule', index=False)
        print(f"Schedule saved to: {output_path}")
    except NameError:
        print("Run the previous schedule-building cell first, then run this cell again.")
else:
    print("No schedule to save (ILP solver did not find a solution).")

Schedule saved to: C:\Work\ML\Graduation\data\Processed Data\Schedule_Output_ILP.xlsx
